In [1]:
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import load_iris
from sklearn.svm import SVC

# 1. Load pre-packaged dataset
iris = load_iris(as_frame=True)
df = iris.frame

# For binary classification clarity: Setosa (0) vs Versicolor (1)
df_binary = df[df["target"].isin([0, 1])].copy()

# Features: Sepal Length, Sepal Width, Petal Length
features = ["sepal length (cm)", "sepal width (cm)", "petal length (cm)"]
X = df_binary[features].values
y = df_binary["target"].values

# 2. Fit a Linear Support Vector Classifier
model = SVC(kernel="linear", C=1.0)
model.fit(X, y)

# Extract weights (coefficients) and intercept: w1*x + w2*y + w3*z + b = 0
w = model.coef_[0]
b = model.intercept_[0]

# 3. Create a 2D mesh grid for the separating hyperplane
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 20), np.linspace(y_min, y_max, 20))

# Solve for z: z = -(w[0]*x + w[1]*y + b) / w[2]
zz = -(w[0] * xx + w[1] * yy + b) / w[2]

# Also calculate parallel margin boundaries (+1 and -1)
zz_margin_up = -(w[0] * xx + w[1] * yy + b - 1) / w[2]
zz_margin_down = -(w[0] * xx + w[1] * yy + b + 1) / w[2]

# 4. Build the Interactive 3D Plot
fig = go.Figure()

# Plot data points for Class 0 (Setosa)
mask_0 = y == 0
fig.add_trace(
    go.Scatter3d(
        x=X[mask_0, 0],
        y=X[mask_0, 1],
        z=X[mask_0, 2],
        mode="markers",
        name="Setosa (Class 0)",
        marker=dict(size=6, color="#1f77b4", symbol="circle"),
    )
)

# Plot data points for Class 1 (Versicolor)
mask_1 = y == 1
fig.add_trace(
    go.Scatter3d(
        x=X[mask_1, 0],
        y=X[mask_1, 1],
        z=X[mask_1, 2],
        mode="markers",
        name="Versicolor (Class 1)",
        marker=dict(size=6, color="#ff7f0e", symbol="circle"),
    )
)

# Highlight Support Vectors
support_vectors = model.support_vectors_
fig.add_trace(
    go.Scatter3d(
        x=support_vectors[:, 0],
        y=support_vectors[:, 1],
        z=support_vectors[:, 2],
        mode="markers",
        name="Support Vectors",
        marker=dict(
            size=9,
            color="rgba(0,0,0,0)",
            line=dict(color="black", width=4),
            symbol="circle-open",
        ),
    )
)

# Add Decision Boundary Surface
fig.add_trace(
    go.Surface(
        x=xx,
        y=yy,
        z=zz,
        name="Decision Hyperplane",
        colorscale=[[0, "gray"], [1, "gray"]],
        showscale=False,
        opacity=0.65,
    )
)

# Layout configuration
fig.update_layout(
    title="Linear SVC: 3D Separating Hyperplane & Support Vectors",
    scene=dict(
        xaxis_title="Sepal Length (cm)",
        yaxis_title="Sepal Width (cm)",
        zaxis_title="Petal Length (cm)",
    ),
    legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, b=0, t=40),
)

fig.show()